In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
import itertools

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean
from torch_scatter import scatter
from typing import Tuple, Optional, Dict, List, Sequence, Any

In [169]:
from torch_pointcloud.layers.blocks import linear_block
from torch_pointcloud.layers.activations import ActLike
from torch_pointcloud.layers.norms import NormLike

linear_block(3, 64, act=nn.ReLU(), norm=nn.BatchNorm1d(64), bias=True)

Sequential(
  (0): Linear(in_features=3, out_features=64, bias=True)
  (1): ReLU()
  (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.0, inplace=False)
)

In [158]:
class TNet(nn.Module):
    def __init__(
        self,
        k: int,
        mlp1_dims: Sequence[int] = (64, 128, 1024),
        mlp2_dims: Sequence[int] = (512, 256),
        act: str = "relu",
        norm: str = "batch_norm1d",
        global_pool: str = "max",
    ) -> None:
        super().__init__()
        self.k = k
        self.global_pool = global_pool

        mlp1_dims = list(mlp1_dims)
        mlp2_dims = list(mlp2_dims)

        blocks = []
        for in_features, out_features in itertools.pairwise([k] + mlp1_dims):
            block = linear_block(in_features, out_features, act=act, norm=norm, dropout=None, order="lan")
            blocks.append(block)
        self.mlp1 = nn.Sequential(*blocks)

        blocks = []
        for in_features, out_features in itertools.pairwise([mlp1_dims[-1]] + mlp2_dims):
            block = linear_block(in_features, out_features, act=act, norm=norm, dropout=None, order="lan")
            blocks.append(block)
        self.mlp2 = nn.Sequential(*blocks)

        self.transform = nn.Linear(mlp2_dims[-1], k * k)
        nn.init.zeros_(self.transform.weight)
        nn.init.eye_(self.transform.bias.view(k, k))

    def forward(self, x: torch.Tensor, batch_idxs: torch.Tensor) -> torch.Tensor:
        x = self.mlp1(x)
        x = scatter(x, batch_idxs, dim=0, reduce=self.global_pool)
        x = self.mlp2(x)

        x = self.transform(x)
        iden = torch.eye(self.k, dtype=x.dtype, device=x.device)
        x = x.view(-1, self.k, self.k) + iden

        return x[batch_idxs]


In [119]:
coords = torch.randn(1000, 3)     # XYZ coordinates
rgb = torch.rand(1000, 3)         # RGB values
# Batch of two point clouds
batch = torch.tensor([0] * 600 + [1] * 400, dtype=torch.long)

tnet = TNet(k=3)
out = tnet(coords, batch)
out.shape

torch.Size([1000, 3, 3])

In [160]:
class PointNetEncoder(nn.Module):
    def __init__(
        self,
        mlp1_dims: Sequence[int] = (3, 64),
        mlp2_dims: Sequence[int] = (64, 128, 1024),
        stnet: Optional[nn.Module] = None,
        ftnet: Optional[nn.Module] = None,
        global_pool: str = "max",
    ) -> None:
        super().__init__()
        self.stnet = stnet
        self.ftnet = ftnet
        self.global_pool = global_pool

        blocks = []
        for in_features, out_features in itertools.pairwise(mlp1_dims):
            blocks.append(linear_block(in_features, out_features, dropout=None, order="lan"))
        self.mlp1 = nn.Sequential(*blocks)
        
        blocks = []
        for in_features, out_features in itertools.pairwise(mlp2_dims):
            blocks.append(linear_block(in_features, out_features, dropout=None, order="lan"))
        self.mlp2 = nn.Sequential(*blocks)
        
    def forward(self, x: torch.Tensor, features: Optional[torch.Tensor], batch_idxs: torch.Tensor) -> Dict[str, Any]:
        if self.stnet is not None:
            xt = self.stnet(x, batch_idxs)
            x = torch.bmm(x.unsqueeze(1), xt).squeeze(1)
            
        if features is not None:
            x = torch.cat([x, features], dim=1)
        
        x = self.mlp1(x)

        if self.ftnet is not None:
            xt = self.ftnet(x, batch_idxs)
            x = torch.bmm(x.unsqueeze(1), xt).squeeze(1)
        
        x = self.mlp2(x)
        
        global_feat = scatter(x, batch_idxs, dim=0, reduce=self.global_pool)     
        return x, global_feat

In [161]:
# Initialize with feature dimensions
encoder = PointNetEncoder(
    mlp1_dims=(6, 64),
    mlp2_dims=(64, 128, 1024),
    stnet=TNet(k=3),
    ftnet=TNet(k=64)
)

# Forward pass with features
x = torch.randn(1000, 3)         # coordinates
features = torch.randn(1000, 3)   # RGB values
batch = torch.tensor([0] * 600 + [1] * 400, dtype=torch.long)

out = encoder(x, features, batch)
out[0].shape, out[1].shape

(torch.Size([1000, 1024]), torch.Size([2, 1024]))

In [177]:

class PointNetEncoder(nn.Module):
    def __init__(
        self,
        coords_dim: int = 3,
        features_dim: int = 0,
        mlp1_dims: Sequence[int] = (64,),
        mlp2_dims: Sequence[int] = (128, 1024),
        act: ActLike = "relu",
        norm: NormLike = "batch_norm1d",
        global_pool: str = "max",
        use_features_transform: bool = True,
        tnet_mlp1_dims: Sequence[int] = (64, 128, 1024),
        tnet_mlp2_dims: Sequence[int] = (512, 256),
        tnet_act: ActLike = "relu",
        tnet_norm: NormLike = "batch_norm1d",
        tnet_global_pool: str = "max",
    ) -> None:
        super().__init__()
        mlp1_dims = [coords_dim + features_dim] + list(mlp1_dims)
        mlp2_dims = [mlp1_dims[-1]] + list(mlp2_dims)

        if mlp1_dims[0] != (coords_dim + features_dim):
            raise ValueError("First dimension of mlp1 must be `coords_dim + features_dim`")
        if mlp1_dims[-1] != mlp2_dims[0]:
            raise ValueError("Last dimension of mlp1 must match first dimension of mlp2")

        self.stnet = TNet(
            k=coords_dim,
            mlp1_dims=tnet_mlp1_dims,
            mlp2_dims=tnet_mlp2_dims,
            act=tnet_act,
            norm=tnet_norm,
            global_pool=tnet_global_pool,
        )

        self.ftnet = None
        if use_features_transform:
            self.ftnet = TNet(
                k=mlp1_dims[-1],
                mlp1_dims=tnet_mlp1_dims,
                mlp2_dims=tnet_mlp2_dims,
                act=tnet_act,
                norm=tnet_norm,
                global_pool=tnet_global_pool,
            )

        blocks = []
        for in_features, out_features in itertools.pairwise(mlp1_dims):
            block = linear_block(in_features, out_features, act=act, norm=norm, dropout=None, order="lan")
            blocks.append(block)
        self.mlp1 = nn.Sequential(*blocks)

        blocks = []
        for in_features, out_features in itertools.pairwise(mlp2_dims):
            block = linear_block(in_features, out_features, act=act, norm=norm, dropout=None, order="lan")
            blocks.append(block)
        self.mlp2 = nn.Sequential(*blocks)

        self.global_pool = global_pool

    def forward(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch_idxs: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        xt = self.stnet(coords, batch_idxs)
        x = torch.bmm(coords.unsqueeze(1), xt).squeeze(1)

        if features is not None:
            x = torch.cat([x, features], dim=1)

        x = self.mlp1(x)

        if self.ftnet is not None:
            xt = self.ftnet(x, batch_idxs)
            x = torch.bmm(x.unsqueeze(1), xt).squeeze(1)

        x = self.mlp2(x)

        global_feat = scatter(x, batch_idxs, dim=0, reduce=self.global_pool)
        return x, global_feat


In [180]:
# Initialize with feature dimensions
encoder = PointNetEncoder(
    coords_dim=3,
    features_dim=10,
    mlp1_dims=(64,),
    mlp2_dims=(128, 1024),
)

# Forward pass with features
x = torch.randn(1000, 3)         # coordinates
features = torch.randn(1000, 10)   # RGB values
batch = torch.tensor([0] * 600 + [1] * 400, dtype=torch.long)

out = encoder(x, features, batch)
out[0].shape, out[1].shape

(torch.Size([1000, 1024]), torch.Size([2, 1024]))

In [181]:
class PointNetClassification(nn.Module):
    def __init__(
        self,
        num_classes: int,
        coords_dim: int = 3,
        features_dim: int = 0,
        dropout: float = 0.3
    ) -> None:
        super().__init__()
        
        # Original PointNet architecture dimensions
        self.encoder = PointNetEncoder(
            coords_dim=coords_dim,
            features_dim=features_dim,
            mlp1_dims=(64,),              # First MLP: 3 -> 64
            mlp2_dims=(128, 1024),        # Second MLP: 64 -> 128 -> 1024
            use_features_transform=True,
            # T-Net architectures (same for both transforms)
            tnet_mlp1_dims=(64, 128, 1024),
            tnet_mlp2_dims=(512, 256),
        )
        
        # Classification head (as per original paper)
        self.classifier = nn.Sequential(
            linear_block(1024, 512, dropout=dropout, order="land"),    # Note the dropout
            linear_block(512, 256, dropout=dropout, order="land"),     # Note the dropout
            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch_idxs: torch.Tensor,
    ) -> torch.Tensor:
        # Get global features from encoder
        _, global_feat = self.encoder(coords, features, batch_idxs)
        
        # Get predictions (one per cloud)
        logits = self.classifier(global_feat)
        
        return logits

def create_pointnet_classification(
    num_classes: int,
    coords_dim: int = 3,
    features_dim: int = 0,
    **kwargs
) -> PointNetClassification:
    return PointNetClassification(
        num_classes=num_classes,
        coords_dim=coords_dim,
        features_dim=features_dim,
        **kwargs
    )

In [184]:
# Create model
model = create_pointnet_classification(
    num_classes=40,  # e.g., for ModelNet40
    coords_dim=3,    # XYZ coordinates
    features_dim=0   # No additional features
)

# Forward pass
coords = torch.randn(1000, 3)  # 1000 points, XYZ coords
batch = torch.tensor([0] * 600 + [1] * 400, dtype=torch.long)  # 2 clouds
logits = model(coords, features=None, batch_idxs=batch)
logits.shape

torch.Size([2, 40])

In [221]:
import functools


def create_pool(reduce: str = "max") -> functools.partial:
    return functools.partial(scatter, reduce=reduce, dim=0)


class PointNetEncoder(nn.Module):
    def __init__(
        self,
        coords_dim: int = 3,
        features_dim: int = 0,
        mlp1_dims: Sequence[int] = (64,),
        mlp2_dims: Sequence[int] = (128, 1024),
        act: ActLike = "relu",
        norm: NormLike = "batch_norm1d",
        global_pool: str = "max",
        use_features_transform: bool = True,
        tnet_mlp1_dims: Sequence[int] = (64, 128, 1024),
        tnet_mlp2_dims: Sequence[int] = (512, 256),
        tnet_act: ActLike = "relu",
        tnet_norm: NormLike = "batch_norm1d",
    ) -> None:
        super().__init__()
        mlp1_dims = [coords_dim + features_dim] + list(mlp1_dims)
        mlp2_dims = [mlp1_dims[-1]] + list(mlp2_dims)

        self.stnet = TNet(
            k=coords_dim,
            mlp1_dims=tnet_mlp1_dims,
            mlp2_dims=tnet_mlp2_dims,
            act=tnet_act,
            norm=tnet_norm,
        )

        self.ftnet = None
        if use_features_transform:
            self.ftnet = TNet(
                k=mlp1_dims[-1],
                mlp1_dims=tnet_mlp1_dims,
                mlp2_dims=tnet_mlp2_dims,
                act=tnet_act,
                norm=tnet_norm,
            )

        blocks = []
        for in_features, out_features in itertools.pairwise(mlp1_dims):
            block = linear_block(in_features, out_features, act=act, norm=norm, dropout=None, order="lan")
            blocks.append(block)
        self.mlp1 = nn.Sequential(*blocks)

        blocks = []
        for in_features, out_features in itertools.pairwise(mlp2_dims):
            block = linear_block(in_features, out_features, act=act, norm=norm, dropout=None, order="lan")
            blocks.append(block)
        self.mlp2 = nn.Sequential(*blocks)

        self.global_pool = global_pool

    def forward(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch_idxs: torch.Tensor,
    ) -> torch.Tensor:
        xt = self.stnet(coords, batch_idxs)
        x = torch.bmm(coords.unsqueeze(1), xt).squeeze(1)

        if features is not None:
            x = torch.cat([x, features], dim=1)

        x = self.mlp1(x)

        if self.ftnet is not None:
            xt = self.ftnet(x, batch_idxs)
            x = torch.bmm(x.unsqueeze(1), xt).squeeze(1)

        x = self.mlp2(x)

        return x


class PointNetClassification(nn.Module):
    def __init__(
        self,
        num_classes: int,
        coords_dim: int = 3,
        features_dim: int = 0,
        dropout: float = 0.0,
        global_pool: str = "max",
        mlp1_dims: Sequence[int] = (64,),
        mlp2_dims: Sequence[int] = (128, 1024),
        act: ActLike = "relu",
        norm: NormLike = "batch_norm1d",
        use_features_transform: bool = True,
        tnet_mlp1_dims: Sequence[int] = (64, 128, 1024),
        tnet_mlp2_dims: Sequence[int] = (512, 256),
        tnet_act: ActLike = "relu",
        tnet_norm: NormLike = "batch_norm1d",
    ) -> None:
        super().__init__()

        self.encoder = PointNetEncoder(
            coords_dim=coords_dim,
            features_dim=features_dim,
            mlp1_dims=mlp1_dims,
            mlp2_dims=mlp2_dims,
            act=act,
            norm=norm,
            use_features_transform=use_features_transform,
            tnet_mlp1_dims=tnet_mlp1_dims,
            tnet_mlp2_dims=tnet_mlp2_dims,
            tnet_act=tnet_act,
            tnet_norm=tnet_norm,
        )

        self.head = nn.Linear(mlp2_dims[-1], num_classes)
        self.global_pool = create_pool(reduce=global_pool)
        self.dropout = dropout

    def forward_features(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch_idxs: torch.Tensor,
    ) -> torch.Tensor:
        return self.encoder(coords, features, batch_idxs)

    def forward_head(self, x: torch.Tensor, batch_idxs: torch.Tensor, pre_logits: bool = False) -> torch.Tensor:
        x = self.global_pool(x, batch_idxs)
        if self.dropout:
            x = F.dropout(x, p=float(self.drop_rate), training=self.training)
        return x if pre_logits else self.head(x)

    def forward(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch_idxs: torch.Tensor,
    ) -> torch.Tensor:
        x = self.forward_features(coords, features, batch_idxs)
        x = self.forward_head(x, batch_idxs)
        return x

In [228]:
# Create model
model = PointNetClassification(
    num_classes=40,  # e.g., for ModelNet40
    coords_dim=3,    # XYZ coordinates
    features_dim=0   # No additional features
)

# Forward pass
coords = torch.randn(1000, 3)  # 1000 points, XYZ coords
batch_idxs = torch.tensor([0] * 600 + [1] * 400, dtype=torch.long)  # 2 clouds
logits = model(coords, features=None, batch_idxs=batch_idxs)
logits.shape

torch.Size([2, 40])

In [229]:
x = model.forward_features(coords, features=None, batch_idxs=batch_idxs)
logits = model.forward_head(x, batch_idxs)
logits.shape

torch.Size([2, 40])

In [193]:
class PointNetSegmentation(nn.Module):
    def __init__(
        self,
        num_classes: int,
        coords_dim: int = 3,
        features_dim: int = 0,
        dropout: float = 0.3
    ) -> None:
        super().__init__()
        
        # Original PointNet segmentation encoder architecture
        self.encoder = PointNetEncoder(
            coords_dim=coords_dim,
            features_dim=features_dim,
            mlp1_dims=(64,),              # First MLP: 3 -> 64
            mlp2_dims=(128, 1024),        # Second MLP: 64 -> 128 -> 1024
            use_features_transform=True,
            tnet_mlp1_dims=(64, 128, 1024),
            tnet_mlp2_dims=(512, 256),
        )
        
        # Segmentation head (as per original paper)
        # Input: point features (128) concatenated with global features (1024)
        self.segmentation_head = nn.Sequential(
            linear_block(1024 + 1024, 512, dropout=dropout, order="land"),
            linear_block(512, 256, dropout=dropout, order="land"),
            linear_block(256, 128, dropout=dropout, order="land"),
            nn.Linear(128, num_classes)
        )

    def forward(
        self,
        coords: torch.Tensor,
        features: Optional[torch.Tensor],
        batch_idxs: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            coords: Point coordinates (N, 3)
            features: Optional point features (N, F)
            batch_idxs: Batch assignments (N,)
        Returns:
            Per-point segmentation logits (N, num_classes)
        """
        # Get point features and global features from encoder
        point_feat, global_feat = self.encoder(coords, features, batch_idxs)
        
        # Expand global features to match point features
        global_feat = global_feat[batch_idxs]
        
        # Concatenate point and global features
        x = torch.cat([point_feat, global_feat], dim=1)  # (N, 128 + 1024)
        
        # Get per-point predictions
        logits = self.segmentation_head(x)
        
        return logits

def create_pointnet_segmentation(
    num_classes: int,
    coords_dim: int = 3,
    features_dim: int = 0,
    **kwargs
) -> PointNetSegmentation:
    """Factory function for creating PointNet segmentation model."""
    return PointNetSegmentation(
        num_classes=num_classes,
        coords_dim=coords_dim,
        features_dim=features_dim,
        **kwargs
    )

In [195]:
# Create segmentation model
model = create_pointnet_segmentation(
    num_classes=50,  # e.g., for part segmentation
    coords_dim=3,    # XYZ coordinates
    features_dim=0   # No additional features
)

# Forward pass
coords = torch.randn(1000, 3)  # 1000 points, XYZ coords
batch = torch.tensor([0] * 600 + [1] * 400, dtype=torch.long)  # 2 clouds
logits = model(coords, features=None, batch_idxs=batch)  # Shape: (1000, num_classes)
logits.shape

torch.Size([1000, 50])